# 01. Apollo Baseline

This notebook runs the full cross-validation benchmark on the purified tumor data.
It implements the exact methodology from the Apollo paper: Feature Selection (Top 20) + SVM.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
import joblib

# Add project root to path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.loader import GliomaDataLoader
from src.models.baselines import ApolloPipeline

In [2]:
# --- Configuration ---
DATA_PATH = "../data/processed/glioma_dataset.h5"
SPLIT_FILE = "../data/splits/manual_splits.json"

if not os.path.exists(SPLIT_FILE):
    print(f"❌ ERROR: Splits file not found at {SPLIT_FILE}")

print("Initializing Loader...")
# Ensure tumor_only=True so we only train on the data identified by our Spatio-Spectral Annotator
loader = GliomaDataLoader(
    data_path=DATA_PATH, 
    split_file=SPLIT_FILE, 
    memory_mapped=False, 
    tumor_only=True
)

print(f"✅ Loader initialized. Total Tumor Spectra: {loader.X.shape[0]}")

Initializing Loader...
📂 Loading data from ../data/processed/glioma_dataset.h5...
🔬 'tumor_only=True': Filtering out healthy tissue...
   ✅ Filtered to 189277 Pure Tumor Spectra.
✅ Loader initialized. Total Tumor Spectra: 189277


## Run Benchmark (5-Fold CV)

In [ ]:
# --- 5-Fold Grouped Cross-Validation ---
# We iterate directly over the loader, which handles the splitting logic internally.

fold_results = []
feature_masks = []

print(f"🚀 Starting Training via Loader Iterator...")

for fold_data in loader:
    fold_id = fold_data['fold_name']
    print(f"\n>>> Running {fold_id} <<<")
    
    # Unpack Data
    X_train, y_train = fold_data['train']
    X_val, y_val = fold_data['val']
    
    print(f"   Train Shape: {X_train.shape}, Val Shape: {X_val.shape}")
    
    # 2. Instantiate Apollo Pipeline 
    # (Re-instantiate per fold to ensure a fresh start)
    model = ApolloPipeline(n_features=20, random_state=42)
    
    # 3. Train
    # This runs: RF Selector -> Top 20 Features -> Scaler -> SVM
    model.fit(X_train, y_train)
    
    # Save the selected features (Critical for Phase 3: TSTR)
    feature_masks.append(model.selected_indices_)
    
    # 5. Evaluate
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    print(f"    > Fold Accuracy: {acc:.4f}")

    os.makedirs("../models/baselines", exist_ok=True)
    model_path = f"../models/baselines/apollo_{fold_id}.joblib"
    joblib.dump(model, model_path)
    print(f"    > Model saved to {model_path}")
    
    # 6. Track Results
    fold_results.append({
        'fold': fold_idx + 1,
        'accuracy': acc,
        'y_true': y_val,
        'y_pred': y_pred
    })
    
    # 7. Track Selected Features
    feature_masks.append(model.selected_indices_)
    
    print(f"   ✅ Fold Accuracy: {acc:.4f}")

# --- Summary ---
mean_acc = np.mean([r['accuracy'] for r in fold_results])
print(f"\n🏆 Average Accuracy: {mean_acc:.4f}")

## Analysis & Reporting

In [ ]:
# Aggregate Accuracy
accuracies = [r['accuracy'] for r in fold_results]
mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)

print(f"\n=== Final Results ===")
print(f"Apollo Baseline Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")

# Create results directory if needed
os.makedirs("../results", exist_ok=True)

# Save simple metrics
metrics_df = pd.DataFrame(fold_results).drop(columns=['y_true', 'y_pred'])
metrics_df.to_csv("../results/baseline_metrics.csv", index=False)
print("✅ Saved clean metrics CSV.")

# Save feature masks for later TSTR evaluation
np.save("../results/feature_masks.npy", feature_masks)
print("✅ Saved selected feature masks to .npy")

In [ ]:
all_true = np.concatenate([r['y_true'] for r in fold_results])
all_pred = np.concatenate([r['y_pred'] for r in fold_results])

cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f"Aggregated Confusion Matrix (Acc: {mean_acc:.4f})")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.show()

In [ ]:
# # Future Usage: To make predictions on new spectra, simply load the trained model and call predict. The pipeline will handle feature selection and scaling automatically.
# loaded_model = joblib.load("../models/baselines/apollo_fold_1.joblib")
# prediction = loaded_model.predict(new_spectrum) 
# # ^ This handles feature selection and scaling automatically!